In [6]:
from langchain_core.runnables.graph_mermaid import draw_mermaid_png
from langgraph.graph import MessagesState, START, END, StateGraph
from langgraph.constants import Send
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.tools import TavilySearchResults
from langchain_community.document_loaders import WikipediaLoader
from langchain_core.messages import get_buffer_string
from typing import List, Annotated
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from dotenv import dotenv_values
from IPython.display import display, Image, Markdown
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import VectorParams, Distance
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import operator
import re

/home/adi-senku/PycharmProjects/FYP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
env_values = {**dotenv_values(".env.shared"), **dotenv_values(".env.secret")}

os.environ["LANGSMITH_TRACING"] = env_values["LANGSMITH_TRACING"]
os.environ["LANGSMITH_ENDPOINT"] = env_values["LANGSMITH_ENDPOINT"]
os.environ["LANGSMITH_API_KEY"] = env_values["LANGSMITH_API_KEY"]
os.environ["LANGSMITH_PROJECT"] = env_values["LANGSMITH_PROJECT"]
os.environ["TAVILY_API_KEY"] = env_values["TAVILY_API_KEY"]
os.environ["OPENAI_API_KEY"] = env_values["OPENAI_API_KEY"]
os.environ['HF_TOKEN'] = env_values['HUGGINGFACE_API_KEY']


In [8]:
print(os.environ.get('LANGSMITH_API_KEY'))

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


In [ ]:
class Analyst(BaseModel):
    affiliation: str = Field(description="Primary affiliation of the analyst.")
    name: str = Field(description="Name of the analyst.")
    role: str = Field(description="Role of the analyst in the context of the topic")
    description: str = Field(description="Description of the analyst focus, concerns and motives.")

    @property
    def persona(self) -> str:
        return f"Name: {self.name}\nRole: {self.role}\nDescription: {self.description}"

class Perspectives(BaseModel):
    analysts: List[Analyst] = Field(description="Comprehensive list of analysts, with their roles and affiliations")

class GenerateAnalystState(TypedDict):
    topic: str # Research topic
    max_analysts: int # No. of analysts
    human_analyst_feedback: str # Human in the loop
    analysts: List[Analyst] # Analyst asking questions


In [ ]:
llm = ChatOpenAI(model="gpt-4o", temperature=0.4, top_p=0.35)

In [ ]:
analyst_instructions = """
You are tasked with creating a set of AI personas. Follow these instructions carefully:
1. First review the research topic:
 {topic}
2. Examine any editorial feedback that has been optionally provided to guide the creation of the analysts:
{human_analyst_feedback}
3. Determine the most interesting themes based upon the documents and / or feedback above.
4. Pick the top {max_analysts} most interesting themes.
5. Assign one analyst per theme.

Output format:
Each analyst must have a role, a name, a affiliation and a designation.
"""

In [ ]:
create_analyst_prompt = ChatPromptTemplate.from_messages([
    ('system', analyst_instructions),
    ("human", "Generate the set of analysts."),
])

In [ ]:
def create_analysts(state: GenerateAnalystState):
    """Create analysts"""
    topic = state["topic"]
    max_analysts = state["max_analysts"]
    human_analyst_feedback = state.get("human_analyst_feedback", "")
    llm_structured = llm.with_structured_output(Perspectives)
    runnable_structured = create_analyst_prompt | llm_structured
    response = runnable_structured.invoke({'topic': topic, 'max_analysts': max_analysts, 'human_analyst_feedback': human_analyst_feedback})
    return {'analysts': response.analysts}

In [ ]:
def human_feedback_node(state: GenerateAnalystState):
    """No-op node, that should be interrupted on"""
    pass

def should_continue(state: GenerateAnalystState):
    """Return to the next node to continue"""
    human_feedback = state.get("human_analyst_feedback", "")
    if human_feedback:
        return "create_analysts"

    return END

In [ ]:
builder = StateGraph(GenerateAnalystState)
builder.add_node("create_analysts", create_analysts)
builder.add_node("human_feedback", human_feedback_node)
builder.add_edge(START, "create_analysts")
builder.add_edge("create_analysts", "human_feedback")
builder.add_conditional_edges("human_feedback", should_continue, ["create_analysts", END])

memory = MemorySaver()
graph = builder.compile(checkpointer=memory, interrupt_after=['human_feedback'])

In [ ]:
display(Image(graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
max_analysts = 3
topic = "Langgraph"
config = {"configurable": {"thread_id": "7"}}

for event in graph.stream({'topic': topic, 'max_analysts': max_analysts}, config=config, stream_mode='values'):
    analysts = event.get('analysts', '')
    if analysts:
        for analyst in analysts:
            print(analyst.persona)
            print("-" * 50)

In [ ]:
graph.update_state(values={'human_analyst_feedback': "Add analysts who are entrepreneur."}, config=config, as_node='human_feedback')

In [ ]:
graph.get_state(config).next

In [ ]:
for event in graph.stream(None, config=config, stream_mode='values'):
    analysts = event.get('analysts', '')
    if analysts:
        for analyst in analysts:
            print("Name: ", analyst.name)
            print("Role: ", analyst.role)
            print("Affiliation:", analyst.affiliation)
            print("Description: ", analyst.description)
            print("-" * 50)

In [ ]:
graph.get_state(config).values.get('analysts', '')

In [ ]:
graph.update_state(config=config, values={'human_analyst_feedback': None}, as_node='human_feedback')

In [ ]:
for event in graph.stream(None, config=config, stream_mode='updates'):
    print("--NODE--")
    # print(event.keys())
    node_name = next(iter(event.keys()))
    print(node_name)

In [ ]:
final_state = graph.get_state(config)

In [ ]:
final_state.next

In [ ]:
analysts = final_state.values.get('analysts')
for analyst in analysts:
    print(analyst.persona)
    print("-" * 50)

## Conduct Interview

In [ ]:
class InterviewState(MessagesState):
    topic: Annotated[list, operator.add]
    max_turns: int
    context: Annotated[list, operator.add]
    analyst: Analyst
    interview: str # Interview transcript
    sections: list # Final key we duplicate in outer state for Send() API

class InterviewStateOutput(MessagesState):
    topic: Annotated[list, operator.add]
    sections: list

class SearchQuery(BaseModel):
    search_query: str = Field(description="Search query for retrieval")

In [ ]:
question_instructions = """You are an analyst tasked with interviewing an expert to learn about a specific topic.

Your goal is boil down to interesting and specific insights related to your topic.

1. Interesting: Insights that are innovative and out of the box and people would not think about.

2. Specific: Insights that avoid generalities and include specific examples from the expert.

This is your persona: {persona}

Begin by introducing yourself using a name that fits your persona, and then ask your question.

Continue to ask questions to drill down and refine your understanding of the topic.

When you are satisfied with your understanding, complete the interview with: "Thank you so much for your help!"

Remember to stay in character throughout your response, reflecting the persona and goals provided to you."""

In [ ]:
generate_question_prompt = ChatPromptTemplate.from_messages([
    ("system", question_instructions),
    # ("placeholder", "{messages}")
])

In [ ]:
def generate_questions(state: InterviewState):
    """ Node to generate questions """
    analyst = state["analyst"]
    messages = state["messages"]
    # if isinstance(messages, dict):
    #     print("dict")
    question = llm.invoke([SystemMessage(content=question_instructions.format(persona=analyst.persona))] + messages)

    return {'messages': [question]}





## Generate Answer: Parallelization

In [ ]:
tavily_search = TavilySearchResults(max_results=3)

In [ ]:
search_instructions = """You will be given a conversation between an analyst and an expert.

Your goal is to generate a well-structured query for use in retrieval and / or web-search related to the conversation.

First, analyze the full conversation.

Pay particular attention to the final question posed by the analyst.

Convert this final question into a well-structured web search query"""

In [ ]:
search_prompt = ChatPromptTemplate.from_messages([
    ("system", search_instructions),
    ("placeholder", "{messages}")
])

In [ ]:
def search_web(state: InterviewState):
    """Retrieve docs from web search"""

    structured_llm = llm.with_structured_output(SearchQuery)
    runnable_structured = search_prompt | structured_llm
    search_query = runnable_structured.invoke({'messages': state['messages']})

    search_docs = tavily_search.invoke(search_query.search_query)

    formatted_search_docs = "\n\n--\n\n".join([
        f'<Document href="{doc["url"]}"/>\n{doc["content"]}\n</Document>'
        for doc in search_docs
    ])
    return {'context': [formatted_search_docs]}

In [ ]:
def search_wikipedia(state: InterviewState):
    """Node to retrieve docs from wikipedia"""
    structured_llm = llm.with_structured_output(SearchQuery)
    runnable_structured = search_prompt | structured_llm
    search_query = runnable_structured.invoke({'messages': state['messages']})

    search_docs = WikipediaLoader(search_query.search_query, load_max_docs=2).load()
    formatted_search_docs = "\n\n--\n\n".join([
        f'<Document source="{doc.metadata["source"]}" page="{doc.metadata.get("page", "")}"/>\n{doc.page_content}\n</Document>'
        for doc in search_docs
    ])

    return {'context': [formatted_search_docs]}


In [ ]:
def search_documents(state: InterviewState):
    last_human_message = state['topic'][-1]
    llm_structured = llm.with_structured_output(RetrieveQuery)
    runnable_query_structured = retrieve_query_prompt | llm_structured
    result = runnable_query_structured.invoke({'prompt': last_human_message})
    query = result.retrieve_query
    docs = retriever.invoke(query)
    context = ""
    subject_written = False
    for doc in docs:
        content = doc.metadata["subject"]
        if content and not subject_written:
            context = context + " " + content
            subject_written = True
        content = doc.page_content
        if content:
            context = context + " " + content

    state['context'] = [context]
    return state


In [ ]:
answer_instructions = """You are an expert being interviewed by an analyst.

Here is analyst area of focus: {goals}.

You goal is to answer a question posed by the interviewer.

To answer question, use this context:

{context}

When answering questions, follow these guidelines:

1. Use only the information provided in the context.

2. Do not introduce external information or make assumptions beyond what is explicitly stated in the context.

3. The context contain sources at the topic of each individual document.

4. Include these sources your answer next to any relevant statements. For example, for source # 1 use [1].

5. List your sources in order at the bottom of your answer. [1] Source 1, [2] Source 2, etc

6. If the source is: <Document source="assistant/docs/llama3_1.pdf" page="7"/>' then just list:

[1] assistant/docs/llama3_1.pdf, page 7

And skip the addition of the brackets as well as the Document source preamble in your citation."""

In [ ]:
generate_answer_prompt = ChatPromptTemplate.from_messages([
    ("system", answer_instructions),
    ("placeholder", "{messages}")
])

In [ ]:
def generate_answer(state: InterviewState):
    """Node to answer the questions from the analyst"""
    analyst = state["analyst"]
    messages = state["messages"]
    context = state["context"]

    # runnable = generate_question_prompt | llm
    # answer = runnable.inovke({'goals': analyst.persona, 'context': context, 'messages': messages})
    answer = llm.invoke([SystemMessage(content=answer_instructions.format(goals=analyst.persona, context=context))] + messages)

    answer.name = "expert"

    return {"messages": [answer]}


In [ ]:
def save_interview(state: InterviewState):
    """Save interviews"""

    messages = state['messages']

    interview = get_buffer_string(messages)

    return {'interview': interview}

In [ ]:
def route_messages(state: InterviewState, name:str = 'expert'):
    """To route messages between question and answer"""
    # This router is used after each question-answer pair
    # This checks if the number of responses exceed the number of turns
    messages = state['messages']
    max_num_turns = state.get("max_num_turns", 2)

    max_responses = len(
        [m for m in messages if isinstance(m, AIMessage) and m.name == name]
    )

    if max_responses >= max_num_turns:
        return 'save_interview'


    # Get the last question answer pair to check if this signals the end of discussion
    last_question = messages[-2]

    if "Thank you so much for your help" in last_question.content:
        return "save_interview"

    return "ask_question"

In [ ]:
section_writer_instructions = """You are an expert technical writer.

Your task is to create a short, easily digestible section of a report based on a set of source documents.

1. Analyze the content of the source documents:
- The name of each source document is at the start of the document, with the <Document tag.

2. Create a report structure using markdown formatting:
- Use ## for the section title
- Use ### for sub-section headers

3. Write the report following this structure:
a. Title (## header)
b. Summary (### header)
c. Sources (### header)

4. Make your title engaging based upon the focus area of the analyst:
The interview is {interview}

5. For the summary section:
- Set up summary with general background / context related to the focus area of the analyst
- Emphasize what is novel, interesting, or surprising about insights gathered from the interview
- Create a numbered list of source documents, as you use them
- Do not mention the names of interviewers or experts
- Aim for approximately 400 words maximum
- Use numbered sources in your report (e.g., [1], [2]) based on information from source documents

6. In the Sources section:
- Include all sources used in your report
- Provide full links to relevant websites or specific document paths
- Separate each source by a newline. Use two spaces at the end of each line to create a newline in Markdown.
- It will look like:

### Sources
[1] Link or Document name
[2] Link or Document name

7. Be sure to combine sources. For example this is not correct:

[3] https://ai.meta.com/blog/meta-llama-3-1/
[4] https://ai.meta.com/blog/meta-llama-3-1/

There should be no redundant sources. It should simply be:

[3] https://ai.meta.com/blog/meta-llama-3-1/

8. Final review:
- Ensure the report follows the required structure
- Include no preamble before the title of the report
- Check that all guidelines have been followed"""


In [ ]:
write_prompt = ChatPromptTemplate.from_messages([
    ("system", section_writer_instructions),
    ("human", "Use this source to write your section: {context}")
])

In [ ]:
def write_section(state: InterviewState):
    """Node to write sections"""

    interview = state["interview"]
    context = state["context"]
    analyst = state["analyst"]

    # print(context)
    # combined = ""
    # for c in context:
    #     combined += c
    # print("COmbined:")
    runnable = write_prompt | llm
    section = runnable.invoke({'interview': interview,  'context': analyst.description})

    return {'sections': [section.content]}

In [ ]:
interview_builder = StateGraph(InterviewState, output=InterviewStateOutput)

# Creating Nodes
interview_builder.add_node("ask_question", generate_questions)
interview_builder.add_node("save_interview", save_interview)
interview_builder.add_node("search_web", search_web)
interview_builder.add_node("search_wikipedia", search_wikipedia)
interview_builder.add_node("search_documents", search_documents)
interview_builder.add_node("answer_question", generate_answer)
interview_builder.add_node("write_section", write_section)

# Creating edges
interview_builder.add_edge(START, "ask_question")
interview_builder.add_edge("ask_question", "search_wikipedia")
interview_builder.add_edge("ask_question", "search_web")
interview_builder.add_edge("ask_question", "search_documents")
interview_builder.add_edge("search_wikipedia", "answer_question")
interview_builder.add_edge("search_web", "answer_question")
interview_builder.add_edge("search_documents", "answer_question")
interview_builder.add_conditional_edges("answer_question", route_messages, ['save_interview', 'ask_question'])
interview_builder.add_edge("save_interview", "write_section")
interview_builder.add_edge("write_section", END)

memory = MemorySaver()
graph = interview_builder.compile(checkpointer=memory).with_config(run_name="Conduct Interviews")

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
analysts[0]

In [ ]:
config = {'configurable': {'thread_id': "23"}}

In [ ]:
interview = graph.invoke({'analyst': analysts[1], "max_num_turns":2, "messages": [("human", "So you said you were writing an article on Langgraph")], 'topic': "Langgraph"}, config=config)

In [ ]:
Markdown(interview['sections'][0])

## Finalize

In [ ]:
class ResearchGraphState(TypedDict):
    topic: Annotated[list, operator.add] # Research topic
    max_analysts: int # Number of analysts
    human_analyst_feedback: str # Human feedback
    analysts: List[Analyst] # Analyst asking questions
    sections: Annotated[list, operator.add]
    introduction: str # Introduction for the final report
    content: str # Content for the final report
    conclusion: str # Conclusion for the final report
    final_report: str # Final report

In [ ]:
def initiate_all_interviews(state: ResearchGraphState):
    """ This is the "map" step where we run each interview sub-graph using Send API """

    # Check if human feedback
    human_analyst_feedback=state.get('human_analyst_feedback')
    if human_analyst_feedback:
        # Return to create_analysts
        return "create_analysts"

    # Otherwise kick off interviews in parallel via Send() API
    else:
        topic = state["topic"][-1]
        return [Send("conduct_interview", {"analyst": analyst,
                                           "messages": [HumanMessage(
                                               content=f"So you said you were writing an article on {topic}?"
                                           )
                                                       ], "topic": [topic]}) for analyst in state["analysts"]]

report_writer_instructions = """You are a technical writer creating a report on this overall topic:

{topic}

You have a team of analysts. Each analyst has done two things:

1. They conducted an interview with an expert on a specific sub-topic.
2. They write up their finding into a memo.

Your task:

1. You will be given a collection of memos from your analysts.
2. Think carefully about the insights from each memo.
3. Consolidate these into a crisp overall summary that ties together the central ideas from all of the memos.
4. Summarize the central points in each memo into a cohesive single narrative.

To format your report:

1. Use markdown formatting.
2. Include no pre-amble for the report.
3. Use no sub-heading.
4. Start your report with a single title header: ## Insights
5. Do not mention any analyst names in your report.
6. Preserve any citations in the memos, which will be annotated in brackets, for example [1] or [2].
7. Create a final, consolidated list of sources and add to a Sources section with the `## Sources` header.
8. List your sources in order and do not repeat.

[1] Source 1
[2] Source 2

Here are the memos from your analysts to build your report from:

{context}"""

def write_report(state: ResearchGraphState):
    # Full set of sections
    sections = state["sections"]
    topic = state["topic"][-1]

    # Concat all sections together
    formatted_str_sections = "\n\n".join([f"{section}" for section in sections])

    # Summarize the sections into a final report
    system_message = report_writer_instructions.format(topic=topic, context=formatted_str_sections)
    report = llm.invoke([SystemMessage(content=system_message)]+[HumanMessage(content=f"Write a report based upon these memos.")])
    return {"content": report.content}

intro_conclusion_instructions = """You are a technical writer finishing a report on {topic}

You will be given all of the sections of the report.

You job is to write a crisp and compelling introduction or conclusion section.

The user will instruct you whether to write the introduction or conclusion.

Include no pre-amble for either section.

Target around 100 words, crisply previewing (for introduction) or recapping (for conclusion) all of the sections of the report.

Use markdown formatting.

For your introduction, create a compelling title and use the # header for the title.

For your introduction, use ## Introduction as the section header.

For your conclusion, use ## Conclusion as the section header.

Here are the sections to reflect on for writing: {formatted_str_sections}"""

def write_introduction(state: ResearchGraphState):
    # Full set of sections
    sections = state["sections"]
    topic = state["topic"][-1]

    # Concat all sections together
    formatted_str_sections = "\n\n".join([f"{section}" for section in sections])

    # Summarize the sections into a final report

    instructions = intro_conclusion_instructions.format(topic=topic, formatted_str_sections=formatted_str_sections)
    intro = llm.invoke([instructions]+[HumanMessage(content=f"Write the report introduction")])
    return {"introduction": intro.content}

def write_conclusion(state: ResearchGraphState):
    # Full set of sections
    sections = state["sections"]
    topic = state["topic"][-1]

    # Concat all sections together
    formatted_str_sections = "\n\n".join([f"{section}" for section in sections])

    # Summarize the sections into a final report

    instructions = intro_conclusion_instructions.format(topic=topic, formatted_str_sections=formatted_str_sections)
    conclusion = llm.invoke([instructions]+[HumanMessage(content=f"Write the report conclusion")])
    return {"conclusion": conclusion.content}

def finalize_report(state: ResearchGraphState):
    """ The is the "reduce" step where we gather all the sections, combine them, and reflect on them to write the intro/conclusion """
    # Save full final report
    content = state["content"]
    if content.startswith("## Insights"):
        content = content.strip("## Insights")
    if "## Sources" in content:
        try:
            content, sources = content.split("\n## Sources\n")
        except:
            sources = None
    else:
        sources = None

    final_report = state["introduction"] + "\n\n---\n\n" + content + "\n\n---\n\n" + state["conclusion"]
    if sources is not None:
        final_report += "\n\n## Sources\n" + sources
    return {"final_report": final_report}

In [ ]:
builder = StateGraph(ResearchGraphState)
builder.add_node("create_analysts", create_analysts)
builder.add_node("human_feedback", human_feedback_node)
builder.add_node("conduct_interview", interview_builder.compile())
builder.add_node("write_report",write_report)
builder.add_node("write_introduction",write_introduction)
builder.add_node("write_conclusion",write_conclusion)
builder.add_node("finalize_report",finalize_report)

# Logic
builder.add_edge(START, "create_analysts")
builder.add_edge("create_analysts", "human_feedback")
builder.add_conditional_edges("human_feedback", initiate_all_interviews, ["create_analysts", "conduct_interview"])
builder.add_edge("conduct_interview", "write_report")
builder.add_edge("conduct_interview", "write_introduction")
builder.add_edge("conduct_interview", "write_conclusion")
builder.add_edge(["write_conclusion", "write_report", "write_introduction"], "finalize_report")
builder.add_edge("finalize_report", END)

# Compile
memory = MemorySaver()
graph = builder.compile(interrupt_before=['human_feedback'], checkpointer=memory)

In [ ]:
display(Image(graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
max_analysts = 3
topic = "What are electrospun nanofibres?"
thread = {"configurable": {"thread_id": "1"}}

# Run the graph until the first interruption
for event in graph.stream({"topic":[topic],
                           "max_analysts":max_analysts},
                          thread,
                          stream_mode="values"):

    analysts = event.get('analysts', '')
    if analysts:
        for analyst in analysts:
            print(analyst.persona)
            print("-"*50)

In [ ]:
graph.update_state(thread, {"human_analyst_feedback":
                                "Add an electrospun nanofibre engineer"}, as_node="human_feedback")

In [ ]:
for event in graph.stream(None,
                          thread,
                          stream_mode="values"):

    analysts = event.get('analysts', '')
    if analysts:
        for analyst in analysts:
            print(analyst.persona)
            print("-"*50)

In [ ]:
graph.update_state(thread, {"human_analyst_feedback":
                            None}, as_node="human_feedback")

In [ ]:
for event in graph.stream(None, thread, stream_mode="updates"):
    print("--Node--")
    node_name = next(iter(event.keys()))
    print(node_name)

In [ ]:
final_state = graph.get_state(thread)
report = final_state.values.get('final_report')
Markdown(report)

## Chatbot

In [ ]:
llm_chat = ChatGoogleGenerativeAI(model="gemini-2.0-flash-001", temperature=0.3, top_p=0.28)


In [ ]:
client = QdrantClient(url='http://localhost:6333')
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:
loader = PyMuPDFLoader(file_path="molecules.pdf", mode="page", extract_tables="markdown")

docs = loader.load()

In [ ]:
docs[0]

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=50)

In [ ]:
docs_split = text_splitter.split_documents(docs)

In [ ]:
if not client.collection_exists(collection_name='pdf-store'):
    client.create_collection(collection_name='pdf-store', vectors_config=VectorParams(size=768, distance=Distance.COSINE))

    vector_store = QdrantVectorStore.from_documents(collection_name='pdf-store', embedding=embeddings, documents=docs_split)
else:
    vector_store = QdrantVectorStore.from_existing_collection(collection_name='pdf-store', embedding=embeddings, url='http://localhost:6333')

In [ ]:
retriever = vector_store.as_retriever(search_type='mmr')

In [ ]:
class ChatbotInput(MessagesState):
    file_path: str
    context: Annotated[list, operator.add]
    topic: str
    output: str
    prompt: str

class ChatbotOutput(MessagesState):
    output: str

class RetrieveQuery(BaseModel):
    retrieve_query: str = Field(description="The query used to retrieve context from the vector stroe")

In [ ]:
def get_last_human_message(messages: list) -> str:
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return message.content

In [ ]:
retrieve_query_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant, who helps in the generation of a query from the prompt. Understand the prompt and generate a query which is same in semantics and will be used to retrieve the context from the vector store. This is the topic: {prompt}"),
    ("human", "Generate the query.")
])

In [ ]:
search_all_prompt = ChatPromptTemplate.from_messages([
    ("system", """
Your goal is to generate a well-structured query for use in retrieval and / or web-search related to the conversation.

First, analyze the full conversation.

Pay particular attention to the prompt posed by the human.

Convert this prompt into a well-structured web search query"""),
    ("placeholder", "{messages}"),
    ("human", "{prompt}")
])

In [ ]:
def search_web_bot(state: ChatbotInput):
    """Retrieve docs from web search"""

    structured_llm = llm.with_structured_output(SearchQuery)
    runnable_structured = search_all_prompt | structured_llm
    search_query = runnable_structured.invoke({'messages': state['messages'], 'prompt': state['prompt']})

    search_docs = tavily_search.invoke(search_query.search_query)

    formatted_search_docs = "\n\n--\n\n".join([
        f'<Document href="{doc["url"]}"/>\n{doc["content"]}\n</Document>'
        for doc in search_docs
    ])
    return {'context': [formatted_search_docs]}

In [ ]:
def search_wikipedia_bot(state: ChatbotInput):
    """Node to retrieve docs from wikipedia"""
    structured_llm = llm.with_structured_output(SearchQuery)
    runnable_structured = search_all_prompt | structured_llm
    search_query = runnable_structured.invoke({'messages': state['messages'], 'prompt': state['prompt']})

    search_docs = WikipediaLoader(search_query.search_query, load_max_docs=2).load()
    formatted_search_docs = "\n\n--\n\n".join([
        f'<Document source="{doc.metadata["source"]}" page="{doc.metadata.get("page", "")}"/>\n{doc.page_content}\n</Document>'
        for doc in search_docs
    ])

    return {'context': [formatted_search_docs]}


In [ ]:
def retrieve_context(state: ChatbotInput):
    last_human_message = state['topic']
    llm_structured = llm.with_structured_output(RetrieveQuery)
    runnable_query_structured = retrieve_query_prompt | llm_structured
    result = runnable_query_structured.invoke({'prompt': last_human_message})
    query = result.retrieve_query
    docs = retriever.invoke(query)
    context = ""
    subject_written = False
    for doc in docs:
        content = doc.metadata["subject"]
        if content and not subject_written:
            context = context + " " + content
            subject_written = True
        content = doc.page_content
        if content:
            context = context + " " + content

    state['context'] = [context]
    return state

In [ ]:
generate_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are an assistant who helps in answering questions, based on the context. Understand the prompt, and answer the question with respect to the context provided. If the prompt provided and the context are not semantically matching, then reply with I don't know the answer.
    Understand the context and then provide the answer. Give a detailed answer.
    This is the context: {context}
    """),
    ("placeholder", "{messages}"),
    ("human", "{input}")
])

In [ ]:
generate_answer_prompt_system_message = """    You are an assistant who helps in answering questions, based on the context. Understand the prompt, and answer the question with respect to the context provided. If the prompt provided and the context are not semantically matching, then reply with an appropriate answer.If the prompt and context is semantically matching then provide answers based on the context.
    Understand the context and then provide the answer. Give a detailed answer.
    This is the context: {context}"""

In [ ]:
def generate(state: ChatbotInput):
    last_human_message = state['prompt']
    runnable_generate = generate_prompt | llm_chat
    result = llm.invoke([SystemMessage(content=generate_answer_prompt_system_message.format(context=state['context'])) ]+ [HumanMessage(content=last_human_message)] + state['messages'])
    state['output'] = result.content
    return state


In [ ]:
builder_chat = StateGraph(ChatbotInput, output=ChatbotOutput)

# Creating Nodes
builder_chat.add_node("retriever", retrieve_context)
builder_chat.add_node("search web", search_web_bot)
builder_chat.add_node("search wikipedia", search_wikipedia_bot)
builder_chat.add_node("answer", generate)

# Creating Edges
builder_chat.add_edge(START, "retriever")
builder_chat.add_edge(START, "search web")
builder_chat.add_edge(START, "search wikipedia")
builder_chat.add_edge(["retriever", "search web", "search wikipedia"], "answer")
builder_chat.add_edge("answer", END)

memory = MemorySaver()
graph_chatbot = builder_chat.compile(checkpointer=memory)

In [ ]:
display(Image(graph_chatbot.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
thread_chat = {"configurable": {"thread_id": "chat_5"}}

In [ ]:
result = graph_chatbot.invoke({'messages': [("human", input())]}, config=thread_chat)

In [ ]:
result['output']

## Final Graph

In [ ]:
class FinalGraphState(MessagesState):
    topic: str
    file_path: str
    final_report: str
    max_analysts: int
    prompt: str
    output: str

In [ ]:
def display_node(state: FinalGraphState):
    if state.get('output', ""):
        display(Markdown(state['output']))
    else:
        display(Markdown(state['final_report']))

In [ ]:
stack_check_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyse the prompt. If the prompt is to write a report, then output 'report', else output 'chatbot'"),
    ("human", "{prompt}")
])

In [ ]:
class StackCheck(BaseModel):
    stack: str = Field(description="The stack to route to")

In [ ]:
def stack_check(state: FinalGraphState):
    human_prompt = state['topic']
    runnable_stack_check = stack_check_prompt | llm.with_structured_output(StackCheck)
    result = runnable_stack_check.invoke({'prompt': human_prompt})
    stack = result.stack
    # print(stack)
    if 'report' in stack:
        return 'report_graph'
    else:
        return 'chatbot_graph'

In [ ]:
final_graph_builder = StateGraph(FinalGraphState)
memory_for_chat_bot = MemorySaver()

final_graph_builder.add_node("display", display_node)
final_graph_builder.add_node("chatbot_graph", builder_chat.compile())
final_graph_builder.add_node("report_graph", builder.compile(interrupt_before=['human_feedback']))

# Create edges
final_graph_builder.add_conditional_edges(START, stack_check, ['chatbot_graph', 'report_graph'])
final_graph_builder.add_edge('chatbot_graph', 'display')
final_graph_builder.add_edge('report_graph', 'display')
final_graph_builder.add_edge('display', END)

memory = MemorySaver()

final_graph = final_graph_builder.compile(checkpointer=memory)

In [ ]:
display(Image(final_graph.get_graph(xray=3).draw_mermaid_png()))

In [ ]:
final_graph_thread = {'configurable': {'thread_id': 'f_32'}}

In [ ]:
prompt = input()
result = final_graph.invoke({'prompt': prompt, 'max_analysts': 2, "topic": [prompt]}, config=final_graph_thread, stream_mode='updates')

In [ ]:
overall_state = final_graph.get_state(final_graph_thread, subgraphs=True)

In [ ]:
report_graph_state = overall_state.tasks[0].state

In [ ]:
report_graph_state.config

In [ ]:
overall_state = final_graph.get_state(final_graph_thread, subgraphs=True)
if "human_feedback" in report_graph_state.next:
    report_config = report_graph_state.config
    hil = input()
    if hil is not None:
        final_graph.update_state(report_config, {"human_analyst_feedback": hil},  as_node="human_feedback")
    else:
        final_graph.update_state(None, config=report_config, as_node="human_feedback")

test = final_graph.invoke(None, config=final_graph_thread)

In [ ]:
test_overall = final_graph.get_state(final_graph_thread, subgraphs=True)

In [ ]:
test_overall.tasks[0].state.next

In [21]:
from langgraph_sdk import get_sync_client
from deployment.states_ds import Analyst
import time
from langgraph.pregel.remote import RemoteGraph
from langgraph.errors import GraphInterrupt
import uuid

In [22]:
url = "http://35.194.18.115:8123"

In [23]:
client = get_sync_client(url=url)

In [24]:
config = {"configurable": {"thread_id": uuid.uuid4()}}

In [25]:
graph = RemoteGraph("nano-ai", sync_client=client, config=config)

In [26]:
graph.get_graph(xray=1).draw_mermaid_png(output_file_path="graph.png")

ReadTimeout: HTTPSConnectionPool(host='mermaid.ink', port=443): Read timed out. (read timeout=10)

In [21]:
user_input = "Hi"

In [22]:
output = graph.invoke(input={'prompt': user_input, 'max_analysts':2, 'topic': [user_input], "messages": [("human", user_input)]}, config=config)

In [23]:
output

{'messages': [{'content': 'Hi',
   'additional_kwargs': {},
   'response_metadata': {},
   'type': 'human',
   'name': None,
   'id': '49a5f580-55c6-4ec4-944e-9aa6a91e9863',
   'example': False},
  {'content': 'Hello! How can I assist you today?',
   'additional_kwargs': {},
   'response_metadata': {},
   'type': 'ai',
   'name': None,
   'id': 'c71e40e6-d52e-4c8f-b916-dfcacc0f0108',
   'example': False,
   'tool_calls': [],
   'invalid_tool_calls': [],
   'usage_metadata': None}],
 'topic': ['Hi'],
 'max_analysts': 2,
 'prompt': 'Hi',
 'output': 'Hello! How can I assist you today?',
 'to_display': 'Hello! How can I assist you today?'}

In [55]:
try:
    output = graph.invoke(input={'prompt': user_input, 'max_analysts':2, 'topic': [user_input], "messages": [("human", user_input)]}, config=config)
except GraphInterrupt:
    state = graph.get_state(config, subgraphs=True)
    report_graph = state.tasks[0].state
    analysts = report_graph.values['analysts']
    report_config = report_graph.config
    print(analysts)
    time.sleep(10)
    hil = input()
    while hil != "":
        graph.update_state(report_config, values={"human_analyst_feedback": hil}, as_node="human_feedback")
        try:
            graph.invoke(None, config=config)
        except GraphInterrupt:
            state = graph.get_state(config, subgraphs=True)
            report_graph = state.tasks[0].state
            analysts = report_graph.values['analysts']
            print(analysts)
            time.sleep(10)
            hil = input()

    graph.update_state(report_config, None, as_node="human_feedback")
    output = graph.invoke(None, config=config)

[{'affiliation': 'University of Computational Linguistics', 'name': 'Dr. Emily Carter', 'role': 'Linguistic Data Analyst', 'description': 'Dr. Emily Carter focuses on the structural and semantic analysis of language graphs. Her primary concern is understanding how different languages can be represented and compared using graph-based models. She is motivated by the potential of langgraph to enhance cross-linguistic studies and improve machine translation systems.'}, {'affiliation': 'Tech Innovations Lab', 'name': 'Rajesh Kumar', 'role': 'Software Engineer and Data Scientist', 'description': 'Rajesh Kumar is dedicated to the technical development and implementation of langgraph systems. His focus is on optimizing algorithms for efficient graph processing and exploring the integration of langgraph with AI applications. Rajesh is driven by the challenge of creating scalable solutions that can handle the complexity of language data in real-time applications.'}]


KeyboardInterrupt: Interrupted by user

In [72]:
report_graph = state.tasks[0].state
analysts = report_graph.values['analysts']
print(analysts)
time.sleep(10)
hil = input()
report_config = report_graph.config
while hil != "":
    graph.update_state(report_config, values={"human_analyst_feedback": hil}, as_node="human_feedback")
    try:
        graph.invoke(None, config=config)
    except GraphInterrupt:
        state = graph.get_state(config, subgraphs=True)
        report_graph = state.tasks[0].state
        analysts = report_graph.values['analysts']
        print(analysts)
        time.sleep(10)
        hil = input()

graph.update_state(report_config, None, as_node="human_feedback")
output = graph.invoke(None, config=config)

[{'affiliation': 'University of Computational Linguistics', 'name': 'Dr. Emily Chen', 'role': 'Linguistic Data Analyst', 'description': 'Dr. Emily Chen focuses on the structural and syntactic aspects of language graphs. Her primary concern is understanding how different languages can be represented and analyzed through graph theory, and how these representations can enhance computational linguistics applications.'}, {'affiliation': 'Global Language Technology Solutions', 'name': 'Raj Patel', 'role': 'Graph Theory Specialist', 'description': 'Raj Patel specializes in the application of graph theory to language processing. His work is centered around optimizing algorithms for language graph construction and exploring their potential in improving machine translation and natural language understanding systems.'}]


In [76]:
Markdown(output["to_display"])

# LangGraph: Revolutionizing AI Agent Development with Cyclic Workflows

## Introduction

LangGraph is a groundbreaking library that enhances large language model (LLM) applications by introducing cyclic computational capabilities, a significant departure from the traditional Directed Acyclic Graphs (DAGs) used in LangChain. This innovation enables the creation of more complex, agent-like behaviors, crucial for applications like interactive chatbots and adaptive learning systems. By allowing LLMs to dynamically loop through processes and adapt based on new information, LangGraph mimics human-like decision-making. This report explores LangGraph's framework for managing multi-agent systems, its impact on AI reasoning capabilities, and its versatility in various applications, positioning it as a major advancement in AI agent development.

---



LangGraph represents a significant advancement in the development of AI agents by introducing cyclic computational capabilities to large language model (LLM) applications. Built on top of LangChain, LangGraph diverges from traditional Directed Acyclic Graphs (DAGs) by allowing the creation of cycles, which enable more complex, agent-like behaviors. This innovation is particularly beneficial for applications that require continuous feedback and adaptive behavior, such as interactive chatbots and adaptive learning systems. By enabling LLMs to dynamically loop through processes, reassess strategies, and modify responses based on new information, LangGraph mimics human-like decision-making processes. This capability is crucial for managing the state of various agents, coordinating their interactions, and handling errors effectively, thereby enhancing flexibility, scalability, and fault tolerance.

LangGraph's versatility extends beyond simple chatbots, making it ideal for complex decision-making tools and dynamic simulation environments. By viewing agent workflows as cyclic graph topologies, LangGraph allows for more variable and nuanced agent behaviors than linear execution models. This positions LangGraph as a major advancement in AI agent development, pushing the limits of what’s possible with AI agents and significantly influencing the future direction of artificial intelligence. The library's cyclic data flows enable systems to learn from past interactions and improve future responses, making it a powerful tool for building stateful, multi-actor applications with LLMs [1][2][3][4][5].

In parallel, graph theory has emerged as a pivotal tool in advancing natural language processing (NLP) and machine translation systems. By leveraging the structural properties of graphs, researchers and developers are optimizing algorithms to enhance the efficiency and accuracy of language processing tasks. Graph theory facilitates clustering and categorization in NLP by representing words or phrases as nodes and their relationships as edges, which is crucial for tasks like topic modeling and document classification [1]. Techniques such as random walks on graphs are employed for tasks like synonym detection, word sense disambiguation, and semantic class detection, utilizing the structure of language graphs to infer semantic relationships [2]. Additionally, graph databases are used for semantic queries in NLP, allowing for efficient querying of relationships between data items, which is particularly beneficial for information retrieval and knowledge graph construction [3].

Despite these advancements, constructing language graphs for machine translation presents challenges, such as accurately representing idiomatic expressions and cultural nuances, and addressing syntactic disambiguation. Automated translation tools are incorporating domain-specific terminology and cultural context into graph structures to address these challenges [4]. Advanced graph-based algorithms are being developed to better capture syntactic dependencies and relationships within sentences [5]. Language graphs are also viewed as complex adaptive systems, where dynamic interactions between components need accurate modeling. By applying principles from complexity science, researchers are creating robust models that adapt to changes, enhancing the translation process [6].

These applications and ongoing research efforts underscore the transformative potential of graph theory in language processing, offering novel solutions to longstanding challenges and paving the way for more sophisticated NLP systems.


---

## Conclusion

LangGraph represents a significant leap forward in AI agent development by introducing cyclic computational capabilities to LLM applications. This innovation allows for more complex, agent-like behaviors, enhancing the reasoning capabilities of AI systems through continuous feedback and adaptive behavior. LangGraph's framework effectively manages state, coordinates interactions, and handles errors in multi-agent systems, making it suitable for diverse applications like adaptive learning systems and interactive chatbots. Meanwhile, the application of graph theory in language processing is revolutionizing NLP and machine translation, optimizing algorithms for clustering, semantic queries, and syntactic disambiguation. Together, these advancements underscore the transformative potential of graph-based approaches in AI, paving the way for more sophisticated and adaptive systems.

## Sources
[1] https://medium.com/@cplog/introduction-to-langgraph-a-beginners-guide-14f9be027141  
[2] https://www.datacamp.com/tutorial/langgraph-tutorial  
[3] https://www.analyticsvidhya.com/blog/2024/07/langgraph-revolutionizing-ai-agent/  
[4] https://becomingahacker.org/a-quick-introduction-to-langgraph-enhancing-llm-applications-with-cyclic-workflows-145f61f38747  
[5] https://adasci.org/a-practical-guide-to-building-ai-agents-with-langgraph/  
[6] https://www.quora.com/What-is-the-application-of-graph-theory-or-network-analysis-in-natural-language-processing-information-retrieval  
[7] https://web.eecs.umich.edu/~mihalcea/papers/nastase.jnle15.pdf  
[8] https://en.wikipedia.org/wiki/Graph_database  
[9] https://ijkcdt.net/_EP/view/?aidx=42660&bidx=3867  
[10] https://ceur-ws.org/Vol-2576/paper08.pdf  
[11] https://en.wikipedia.org/wiki/Complex_adaptive_system

In [25]:
graph.get_state(config)

StateSnapshot(values={'messages': [{'content': 'Hi', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': '65bc44c4-2f2f-48f9-8dd5-a4f901771b52', 'example': False}, {'content': 'Hello! How can I assist you today?', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'ai', 'name': None, 'id': '12c74316-33f6-4019-9104-229062cb76a6', 'example': False, 'tool_calls': [], 'invalid_tool_calls': [], 'usage_metadata': None}], 'topic': ['Hi'], 'max_analysts': 2, 'prompt': 'Hi', 'output': 'Hello! How can I assist you today?', 'to_display': 'Hello! How can I assist you today?'}, next=(), config={'configurable': {'thread_id': 'bdd70a2f-ded2-440c-98c2-7231934ee033', 'checkpoint_ns': '', 'checkpoint_id': '1f00f075-4b03-6ff4-8002-94bbdfed8848', 'checkpoint_map': {}}}, metadata={'user-agent': 'langgraph-sdk-py/0.1.51', 'langgraph_auth_user': None, 'langgraph_auth_user_id': '', 'langgraph_auth_permissions': [], 'graph_id': 'nano-ai', 'assistant_id': '06a9b197-7d05

In [14]:
output

{'messages': [{'content': 'Hi',
   'additional_kwargs': {},
   'response_metadata': {},
   'type': 'human',
   'name': None,
   'id': 'dc64f79e-3131-4d15-85cb-83e59bc8c567',
   'example': False},
  {'content': 'Hello! How can I assist you today?',
   'additional_kwargs': {},
   'response_metadata': {},
   'type': 'ai',
   'name': None,
   'id': '04491222-5db5-4a65-9a31-acee4091974a',
   'example': False,
   'tool_calls': [],
   'invalid_tool_calls': [],
   'usage_metadata': None}],
 'topic': ['Hi'],
 'max_analysts': 2,
 'prompt': 'Hi',
 'output': 'Hello! How can I assist you today?',
 'to_display': 'Hello! How can I assist you today?'}

In [29]:
# Logic for human in the loop to update the type of experts in conversation

state =  client.threads.get_state(thread['thread_id'], subgraphs=True)
check_state_report = state['tasks'][0]['state']
hil = state['tasks'][0]['state']['next'][-1]
config = check_state_report.get("checkpoint")
analysts = check_state_report['values']['analysts']
while hil == 'human_feedback':
    print(analysts)
    time.sleep(5)
    user_input_hil = input()
    checkpoint = client.threads.update_state(values={'human_analyst_feedback': user_input_hil, 'analysts': analysts}, as_node="human_feedback", thread_id=config['thread_id'], checkpoint=config)

    output = client.runs.wait(config['thread_id'], "nano-ai", input=None, checkpoint=checkpoint['checkpoint'])
    # output = client.runs.join(thread['thread_id'], run['run_id'])
    # user_input_hil = input()
    check_state_report = client.threads.get_state(thread['thread_id'], subgraphs=True, checkpoint=checkpoint['checkpoint'])
    print(check_state_report)
    config = check_state_report['checkpoint']
    # break
    if check_state_report.get("next")[0] == "create_analysts":
        analysts = check_state_report['tasks'][0]['result']['analysts']
        continue
    else:
        break




[{'affiliation': 'University of Linguistics', 'name': 'Dr. Emily Chen', 'role': 'Linguistic Data Analyst', 'description': 'Dr. Emily Chen focuses on the structural and syntactic analysis of language graphs. Her primary concern is understanding how different languages can be represented and compared using graph theory, and she is motivated by the potential of these representations to enhance computational linguistics and language processing technologies.'}, {'affiliation': 'Tech Innovations Lab', 'name': 'Raj Patel', 'role': 'Computational Linguist', 'description': 'Raj Patel is dedicated to exploring the applications of language graphs in machine learning and AI. His focus is on how these graphs can improve natural language understanding and generation in AI systems. He is particularly interested in the practical implications of langgraph in developing more intuitive and context-aware AI applications.'}]
{'values': {'messages': [{'content': 'Write a report on langgraph', 'additional_kw

In [ ]:
check_state_report

In [30]:
Markdown(output['to_display'])

# LangGraph: A Paradigm Shift in AI Development

## Introduction

LangGraph is revolutionizing the field of natural language processing (NLP) by introducing a graph-based framework that transcends traditional statistical methods. This report explores how LangGraph enhances AI-driven applications, particularly in multilingual contexts, by enabling dynamic and flexible interactions. Key features include **Flow Engineering**, which allows iterative and controllable interactions with large language models (LLMs), and **Stateful Workflows**, which support complex, parallel processing in multi-agent systems. By structuring applications as graphs, LangGraph facilitates efficient workflows, offering new possibilities for multilingual and complex task management. This report delves into LangGraph's transformative impact on AI development, highlighting its potential in various domains.

---



LangGraph emerges as a transformative tool in the realm of natural language processing (NLP) and artificial intelligence (AI) development, offering a graph-based approach that significantly diverges from traditional statistical methods. This innovative framework is particularly adept at enhancing AI-driven applications in multilingual contexts, enabling more dynamic and flexible interactions.

A pivotal feature of LangGraph is its facilitation of **Flow Engineering**, which allows for an iterative and controllable approach to interacting with large language models (LLMs). Unlike traditional single-prompt interactions, LangGraph supports an iterative "flow" where the LLM is queried in a loop, influencing subsequent actions and yielding superior results [1]. This iterative process enhances the responsiveness and adaptability of AI systems, making them more effective in handling complex tasks.

LangGraph also excels in supporting **Stateful Workflows**. Traditional multi-agent systems often operate in a linear and isolated manner, with each agent handling distinct steps of a workflow sequentially. In contrast, LangGraph orchestrates more complex workflows with inter-agent dependencies, allowing for parallel processing and scalability. This enables agents to run in parallel, improving efficiency, and allows for the addition of new agents to handle tasks such as filtering, data enrichment, or nuanced analyses [2].

The graph-based structure of LangGraph facilitates the development of multi-agent applications that leverage the power of LLMs. By structuring applications as graphs, where nodes represent actions and edges represent relationships, developers can create more organized and efficient workflows. This approach is particularly beneficial in multilingual contexts, where LangGraph can filter and summarize data in multiple languages, then aggregate these findings into a cohesive final report [3].

LangGraph's ability to handle cyclic data flows is essential for most agentic architectures. This capability, combined with its core benefits of cycles, controllability, and persistence, makes LangGraph a valuable tool for developers aiming to create sophisticated, intelligent, and responsive applications. By integrating with existing tools, LangGraph expands the possibilities for developers working with language models, enabling the creation of more interactive and agent-like behaviors [2][3].

In specific applications such as nanotechnology, LangGraph's capabilities can be leveraged to improve data analysis and material design. It can manage complex data workflows from simulations and experiments, facilitating more efficient data processing and insights extraction. Additionally, LangGraph can support simulation workflows in material design, allowing researchers to refine models based on previous results and interactions. Its ability to maintain conversations and remember past interactions can also be used to develop interactive applications for hypothesis testing and providing recommendations based on previous experiments [1][3].

Overall, LangGraph represents a significant advancement in the development of interactive applications using language models, offering fresh opportunities for developers to craft more sophisticated and responsive applications.


---

## Conclusion

LangGraph emerges as a transformative tool in the realm of AI development, offering a graph-based approach that significantly enhances natural language processing capabilities. By enabling **Flow Engineering**, LangGraph allows for iterative interactions with large language models, leading to more dynamic and adaptable AI systems. Its support for **Stateful Workflows** further distinguishes it by orchestrating complex, parallel processes that improve efficiency and scalability. The library's ability to handle cyclic data flows and maintain conversational context makes it invaluable for developing sophisticated multi-agent applications. Overall, LangGraph revolutionizes the development of AI-driven applications, particularly in multilingual and complex task environments, paving the way for more intelligent and responsive systems.

## Sources
[1] https://adasci.org/a-practical-guide-to-building-ai-agents-with-langgraph/  
[2] https://insights.encora.com/insights/building-multi-agent-llms-traditional-approach-vs-langgraph  
[3] https://www.restack.io/p/multi-agents-answer-langgraph-example-cat-ai  
[4] https://langchain-ai.github.io/langgraph/tutorials/introduction/  
[5] https://github.com/langchain-ai/langgraph  
[6] https://www.marktechpost.com/2024/01/27/meet-langgraph-an-ai-library-for-building-stateful-multi-actor-applications-with-llms-built-on-top-of-langchain/

In [ ]:
output['to_display']

In [ ]:
await client.runs.list(thread['thread_id'])

In [ ]:
await client.runs.join('58519888-e395-45bc-846f-bf766762ecaf',  '1f00cc7e-f384-6095-9ae9-4168b2d6ef0c')

In [4]:
import pymongo
import gridfs

In [10]:
client = pymongo.MongoClient("mongodb://localhost:27017/")

In [12]:
db = client["test"]
file_upload = gridfs.GridFS(db, "test")

In [6]:
collection = db.create_collection("test")

In [7]:
collection.insert_one({'test': "1234"})

InsertOneResult(ObjectId('67ed526ebe5913b98ed510f3'), acknowledged=True)

In [9]:
client.drop_database("test")

In [13]:
with open("molecules.pdf", "rb") as f:
    data = f.read()

In [17]:
file_upload.put(data, filename="molecules.pdf", processed=False)

ObjectId('67ed5512be5913b98ed51105')

In [26]:
next(file_upload.find({'processed': False}))._id

ObjectId('67ed5512be5913b98ed51105')

In [36]:
file_collection = db["test.files"]

In [41]:
files = file_collection.find({'processed': False})

In [42]:
for file in files:
    # print(file._id)
    print(file["_id"])
    file_collection.update_one(
        {"_id": file["_id"]},
        {"$set": {"processed": True}},
    )

67ed5512be5913b98ed51105
